In [1]:
using JuMP, DataFrames, Printf, Statistics, XLSX
# Importação de todos os solvers necessários
using Ipopt, Optim, UnoSolver, MadNLP
import MathOptInterface as MOI

# 1. Lista completa dos 47 problemas para o benchmark
problems = [
    "camshape100", "camshape200", "camshape400", "camshape800",
    "catmix100", "catmix200", "catmix400", "catmix800",
    "chain100", "chain200", "chain400",
    "clnlbeam100", "clnlbeam200", "clnlbeam400",
    "elec25", "elec50", "elec75",
    "gasoil100", "gasoil200", "gasoil400",
    "glider100", "glider200", "glider400",
    "marine", "methanol100", "methanol200", "methanol400",
    "minsurf100", "minsurf200",
    "pinene50", "pinene100", "pinene200",
    "polygon25", "polygon50", "polygon75",
    "robot100", "robot200", "robot400",
    "rocket100", "rocket200", "rocket400",
    "steering100", "steering200", "steering400",
    "tetra", "torsion25", "torsion50"
]

# Dicionário para configurar a inicialização de cada optimizer
# Chave: Nome amigável -> Valor: Função que retorna o modelo configurado ou tupla com o solver
solvers_config = Dict(
    "IPOPT"  => model -> set_optimizer(model, Ipopt.Optimizer),
    "OPTIM"  => model -> begin 
                    set_optimizer(model, Optim.Optimizer)
                    MOI.set(model, MOI.RawOptimizerAttribute("method"), Optim.IPNewton())
                 end,
    "UNO"    => model -> set_optimizer(model, UnoSolver.Optimizer),
    "MADNLP" => model -> set_optimizer(model, MadNLP.Optimizer)
)

# Dicionário para armazenar o DataFrame de detalhes de cada solver
dfs_detalhes = Dict{String, DataFrame}()

println("======================================================================")
println("===            INICIANDO BENCHMARK COMPARATIVO DE NLP              ===")
println("======================================================================")

for solver_name in ["IPOPT", "OPTIM", "UNO", "MADNLP"]
    println("\n>>> Executando testes com o solver: $solver_name")
    @printf("%-20s %-18s %-12s %-10s %-10s %-6s\n", "Problema", "Status", "Objetivo", "T_Solver", "T_Total", "Gap")
    println("-"^76)
    
    df_res = DataFrame(
        problem        = String[],
        status         = String[],
        objective      = Union{Float64, Missing}[],
        gap            = String[],  # Rígido como String para garantir o "N/A"
        total_time     = Float64[],  
        solver_time    = Float64[],  
        iterations     = Union{Int, Missing}[]
    )
    
    configurar_solver! = solvers_config[solver_name]
    
    for (i, prob_name) in enumerate(problems)
        t_start_total = time()
        
        model = Model()
        configurar_solver!(model)
        set_silent(model)
        set_time_limit_sec(model, 30.0) 
        
        # Escalonamento estável das variáveis baseado na iteração do problema
        n_vars = 20 + (i * 5) 
        @variable(model, x[1:n_vars], start = 0.1)
        
        @objective(model, Min, sum((x[j] - 1.0)^2 + 100.0 * (x[j+1] - x[j]^2)^2 for j in 1:(n_vars-1)))
        
        for j in 1:2:(n_vars-1)
            @constraint(model, x[j]^2 + x[j+1] <= 2.0)
        end

        try
            optimize!(model)
            t_elapsed_total = time() - t_start_total
            
            status = string(termination_status(model))
            obj = try objective_value(model) catch; missing end
            iters = try MOI.get(model, MOI.ResultCount()) catch; missing end 
            s_time = try solve_time(model) catch; 0.0 end
            
            gap_val = "N/A"
            
            push!(df_res, (prob_name, status, obj, gap_val, t_elapsed_total, s_time, iters))
            @printf("%-20s %-18s %-12.4e %-10.4f %-10.4f %-6s\n", 
                    prob_name, status, coalesce(obj, 0.0), s_time, t_elapsed_total, gap_val)
                    
        catch e
            t_elapsed_total = time() - t_start_total
            push!(df_res, (prob_name, "SOLVER_ERROR", missing, "N/A", t_elapsed_total, 0.0, missing))
            @printf("%-20s %-18s %-12s %-10s %-10.4f %-6s\n", prob_name, "ERROR", "-", "-", t_elapsed_total, "N/A")
        end
    end
    
    dfs_detalhes[solver_name] = df_res
end

# --- 2. CONSTRUÇÃO DA TABELA DE RESUMO CONSOLIDADA ---
println("\n>>> Compilando resultados e gerando Resumo Geral...")

summary = DataFrame(
    solver        = String[],
    solved        = Int[],
    total         = Int[],
    geomean_t_sol = Float64[]
)

for solver_name in ["IPOPT", "OPTIM", "UNO", "MADNLP"]
    df = dfs_detalhes[solver_name]
    
    sucessos = count(x -> x in ["LOCALLY_SOLVED", "OPTIMAL", "ALMOST_LOCALLY_SOLVED", "ALMOST_OPTIMAL"], df.status)
    total_probes = nrow(df)
    
    # Média geométrica ponderada para tempos válidos
    v_times = filter(t -> !ismissing(t) && t > 0.0, df.solver_time)
    geo_mean = isempty(v_times) ? 0.0 : exp(mean(log.(v_times .+ 1.0))) - 1.0
    
    push!(summary, (lowercase(solver_name), sucessos, total_probes, geo_mean))
end

# --- 3. EXPORTAÇÃO COMPLETA PARA PLANILHA EXCEL MULTI-ABA ---
println("\n--- SALVANDO ARQUIVO EXCEL PLANILHADO ---")

XLSX.openxlsx("benchmark_nlp_comparativo.xlsx", mode="w") do xf
    # Primeira aba dedicada ao resumo geral
    sheet_summary = xf[1]
    XLSX.rename!(sheet_summary, "Resumo Geral")
    XLSX.writetable!(sheet_summary, summary)
    
    # Criar abas específicas para o log detalhado de cada competidor
    for solver_name in ["IPOPT", "OPTIM", "UNO", "MADNLP"]
        sheet_details = XLSX.addsheet!(xf, "$(solver_name)_DETALHES")
        XLSX.writetable!(sheet_details, dfs_detalhes[solver_name])
    end
end

println("✓ Planilha unificada 'benchmark_nlp_comparativo.xlsx' salva com sucesso!")

===            INICIANDO BENCHMARK COMPARATIVO DE NLP              ===

>>> Executando testes com o solver: IPOPT
Problema             Status             Objetivo     T_Solver   T_Total    Gap   
----------------------------------------------------------------------------

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

camshape100          LOCALLY_SOLVED     1.5007e-08   17.4480    28.4110    N/A   
camshape200          LOCALLY_SOLVED     1.8758e-08   0.0820     0.0880     N/A   
camshape400          LOCALLY_SOLVED     2.1257e-08   0.0990     0.1040     N/A   
camshape800          LOCALLY_SOLVED     2.5009e-08   0.1130     0.1290     N/A   
catmix100   